# COSC325 Homework 3
Benjamin Belandres, bbelandr

### Instructions
Homework #3 is due November 12, 2025, 11:59 pm.The late deadline is November 14, 2025, 11:59 pm.
* Read each problem's instructions carefully and provide concise answers. 
* Share your Python code in a fully functional Jupyter NotebookHow to share code?Zip your Jupyter Notebook and Python files (You may not have Python files)
* Upload to Canvas. 
* You will also upload a PDF of the notebook to Canvas to ensure that cell output is preserved.   
* The write-up should be submitted as a PDF file.
* In total, there are three files you should submit: Zipped Jupyter/Python code. 
* A PDF file of the Jupyter Notebook and all its output. 
* A PDF with your write-up.
* You are responsible for verifying that all submitted files are complete and represent your latest and greatest work. Any changes to files after the deadline will result in the corresponding penalties.
* A ten-point penalty will be applied to your score for each day after the submission deadline. Automated zero score after the late deadline.


# Task 1: Create a Custom PyTorch Dataset Class to Ingest the Titanic Dataset (25 pts)

* Build a PyTorch Dataset class named TitanicDataset. The dataset should ingest Kaggle's Titanic DatasetTitanic - Machine Learning from Disaster
* Dataset preprocessing steps (e.g., feature generation, imputation) can occur outside the class. (Look into previous notebooks for preprocessing steps.) Transformations that are required inside the class are: 
    * One-hot-encoding
    * Feature scaling
    * Convert to Tensor
* Produce TitanicDataset objects for training and validation datasets
* Ensure no data leakage. 
* Write-up (Under 500 words not including figures)Describe your pre-processing steps and justify your feature engineering decisions. (Under 200 words)Include a dataflow diagram (hand-drawn or graphic tool based) that outlines all preprocessing steps, indicating where data statistics, such as the mean and standard deviation (used for data standardization), are calculated and applied.
    * What resources did you use to build the TitanicDataset class? Were these resources the most helpful and inspiring to you? (Under 200 words)

### Building a PyTorch Dataset class
Following documentation from the official PyTorch [webpage](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset).

First, we need to define a class that inherits from the PyTorch dataset class. According to documentation, we probably need to overwrite these functions:
* __getitem__(): fetches data according to a given key
* __len__(): returns the length of the dataset (optional)
    * This needs to be done if we end up using Sampler or DataLoader
* __getitems__(): fetches multiple data instances (optional)
    * This can be done to make things faster

The class also needs to be able to take in data from the csv files. This will be done in the __init__() function.
* Use pandas to read the csv file

##### One-hot encoding
This will be done through a one-hot encode function. It returns the modified dataframe and also modifies the dataframe that it holds internally.
```
oneHotEncode(columnNames)
    newDf = df
    for each name in columnNames:
        set uniqueData
        for each data in df[name]: 
            uniqueData.add(data)
            This will give us unique entries for every different input in the column
        remove one of the entries from uniqueData
            This will ensure that we don't over-encode
        for data in uniqueData
            newDf.addColumn(data)
                Make sure this is initialized to 0
        for row in df:
            if row[name] in data:
                newDf[row[name]] = 1
        delete column df[name]
    return newDf
```
##### Min-Max scaling
This will scale all of the columns listed using min-max normalization. It returns the modified dataframe and also modifies the dataframe that it holds internally.

xScaled = (x-xMin)/(xMax - xMin)

##### Standardization
What Min-Max does but with a different formula

z = (x - mean)/standardDeviation

In [ ]:
# !pip install torch
# !pip install pandas

import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

class TitanicDataset(Dataset):
    def __init__(self, pathToDataset, labelColumn):
        self.df = pd.read_csv(pathToDataset)
        self.labelColumn = labelColumn

    # Returns the sample and the sample's label at the index
    # This might need to be returned as a tensor
    def __getitem__(self, index):
        sample = self.df.iloc[index]
        label = sample[self.labelColumn]
        sample = sample.drop(self.labelColumn)
        return sample, label
    
    def __len__(self):
        return len(self.df)
    
    def oneHotEncode(self, columnNames):
        newDf = self.df.copy()
        for name in columnNames:
            # get unique categories (preserve stable order), skip NaN
            cats = list(pd.Series(self.df[name].dropna().unique()))
            if not cats:
                continue
            encodedCats = cats[1:]  # drop the first category to avoid perfect multicollinearity
            for cat in encodedCats:
                colName = f"{name}.{cat}"
                newDf[colName] = 0  # initialize column to 0
                newDf.loc[self.df[name] == cat, colName] = 1    # set 1 where the original column equals this category

            newDf.drop(columns=[name], inplace=True)
        self.df = newDf
        return newDf

    # Returns the modified dataframe and also modifies the dataframe that it holds internally. 
    def minMaxScale(self, columnNames):
        newDf = self.df.copy()

        for name in columnNames:
            # Normalize
            col = self.df[name]
            newDf[name] = (col - col.min()) / (col.max() - col.min())
        self.df = newDf
        return newDf
    
    def standardize(self, columnNames):
        newDf = self.df.copy()

        for name in columnNames:
            # Normalize
            col = self.df[name]
            newDf[name] = (col - col.mean()) / col.std()
        self.df = newDf
        return newDf

    def produceCSV(self, fileName):
        self.df.to_csv(fileName)

    def toTensor(self):
        # Convert DataFrame to numpy array (float32), then to tensor
        tensor = torch.from_numpy(self.df.values.astype(np.float32))
        return tensor

### Preprocessing
Preprocessing is done in the preprocess.py file.

